In [ ]:
import gradio as gr

# État global pour la session (utilisateurs simulés)
utilisateurs = {"test": {"mdp": "1234", "mail": "test@example.com"}}
state = gr.State(False)  # False = non connecté

def login(pseudo, mdp, etat):
    if pseudo in utilisateurs and utilisateurs[pseudo]["mdp"] == mdp:
        return gr.Tabs(selected=1), True, gr.Markdown("Connexion réussie !"), gr.update(visible=False)
    return gr.Tabs(selected=0), etat, gr.Markdown("❌ Identifiants incorrects"), gr.update(visible=True)

def show_register():
    return gr.Tabs(selected=2), state

def register(pseudo, mail, mdp, etat):
    if pseudo in utilisateurs:
        return gr.Tabs(selected=2), etat, gr.Markdown("❌ Pseudo déjà utilisé")
    utilisateurs[pseudo] = {"mdp": mdp, "mail": mail}
    return gr.Tabs(selected=1), True, gr.Markdown("✅ Compte créé ! Redirection...")

def logout(etat):
    utilisateurs.clear()
    return False, gr.Tabs(selected=0)

with gr.Blocks(title="Mon App Sécurisée") as demo:
    onglets = gr.Tabs(selected=0)
    
    with gr.TabItem("Connexion", id=0):
        gr.Markdown("### Connexion")
        pseudo = gr.Textbox(label="Pseudo", placeholder="Entrez votre pseudo")
        mdp = gr.Textbox(label="Mot de passe", type="password")
        btn_login = gr.Button("Connexion", variant="primary")
        status = gr.Markdown(visible=False)
        
    with gr.TabItem("Accueil", id=1, visible=False):
        gr.Markdown("# 🏠 Page d'accueil\nBienvenue !")
        btn_logout = gr.Button("Déconnexion", variant="stop")
    
    with gr.TabItem("Création de compte", id=2):
        gr.Markdown("### Création de compte")
        reg_pseudo = gr.Textbox(label="Pseudo")
        reg_mail = gr.Textbox(label="Email")
        reg_mdp = gr.Textbox(label="Mot de passe", type="password")
        btn_register = gr.Button("Créer mon compte", variant="secondary")
        reg_status = gr.Markdown()
    
    # Événements
    btn_login.click(
        login, inputs=[pseudo, mdp, state], 
        outputs=[onglets, state, status, gr.TabItem(visible=False)]
    )
    btn_register.click(
        register, inputs=[reg_pseudo, reg_mail, reg_mdp, state],
        outputs=[onglets, state, reg_status]
    )
    gr.Button("Créer un compte", elem_classes="my-create-btn").click(
        show_register, outputs=[onglets, state]
    )
    btn_logout.click(logout, state, [state, onglets])

demo.launch()


In [ ]:
import gradio as gr
from PIL import Image
import io
import re
from datetime import datetime

# ===============================================
# DONNÉES DE TEST (remplace DB)
# ===============================================
utilisateurs = {"test": {"mdp": "1234", "mail": "test@example.com"}}

# OCR simulé pour tests (remplace pytesseract)
def ocr_simule(image):
    """Simule l'extraction de texte d'une image pour tests"""
    if image is None:
        return "Aucune image chargée"
    
    # Texte de test simulé (à remplacer par pytesseract.image_to_string(image))
    texte_test = """
    FACTURE #12345
    Date: 15/12/2025
    Client: Jean Dupont
    Produits:
    - Livre Python: 25€
    - Formation IA: 150€
    Total: 175€
    Société: TechFormations SARL
    """
    return texte_test

# Résumé simulé (remplace transformers)
def generer_resume(texte):
    """Simule un résumé intelligent"""
    if len(texte) < 50:
        return texte
    
    # Extraction mots-clés simple
    mots_cles = re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', texte)
    phrases = re.split(r'[.\n]', texte)
    
    resume = f"Document contenant: {', '.join(mots_cles[:5])}. Montant total: 175€."
    return resume

# NER simulé
def extraire_entites(texte):
    """Simule extraction d'entités nommées"""
    entites = []
    
    # Dates
    dates = re.findall(r'\d{1,2}/\d{1,2}/\d{4}', texte)
    for date in dates:
        entites.append({"entity": "DATE", "word": date, "score": 0.95})
    
    # Noms propres (mots avec majuscules)
    noms = re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', texte)
    for nom in noms[:3]:
        entites.append({"entity": "PERSON/ORG", "word": nom, "score": 0.90})
    
    # Montants
    montants = re.findall(r'\d+€', texte)
    for montant in montants:
        entites.append({"entity": "MONEY", "word": montant, "score": 0.98})
    
    return entites

# ===============================================
# FONCTIONS AUTHENTIFICATION
# ===============================================
def verifier_connexion(pseudo, mdp):
    """Vérifie identifiants"""
    if pseudo in utilisateurs and utilisateurs[pseudo]["mdp"] == mdp:
        return True, f"✅ Utilisateur **{pseudo}** connecté"
    return False, "❌ Identifiants incorrects"

def creer_compte(pseudo, mail, mdp, mdp_confirm):
    """Crée nouvel utilisateur"""
    if pseudo in utilisateurs:
        return gr.update(open=True), "❌ Pseudo déjà utilisé"
    if any(u["mail"] == mail for u in utilisateurs.values()):
        return gr.update(open=True), "❌ Email déjà utilisé"
    if mdp != mdp_confirm:
        return gr.update(open=True), "❌ Mots de passe différents"
    
    utilisateurs[pseudo] = {"mdp": mdp, "mail": mail}
    return gr.update(open=False), "✅ Compte créé ! Connectez-vous."

def deconnexion(etat):
    return False, "👋 Déconnecté"

# ===============================================
# FONCTION TRAITEMENT PRINCIPALE
# ===============================================
def traiter_document(image):
    """Pipeline complet: OCR → Résumé → NER"""
    if image is None:
        return "Aucune image", "Chargez une image", []
    
    # 1. OCR
    texte = ocr_simule(image)
    
    # 2. Résumé
    resume = generer_resume(texte)
    
    # 3. Entités
    entites = extraire_entites(texte)
    
    return texte, resume, entites

# ===============================================
# INTERFACE GRADIO
# ===============================================
with gr.Blocks(title="📄 Extraction & Résumé Intelligent") as demo:
    
    # États
    etat_connexion = gr.State(False)
    pseudo_actuel = gr.State("")
    
    gr.Markdown("# 📄 Extraction de Documents & Résumé Intelligent")
    gr.Markdown("**Test**: pseudo `test` / mdp `1234`**")
    
    # ===============================================
    # CONNEXION
    # ===============================================
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🔐 Connexion")
            pseudo_input = gr.Textbox(label="Pseudo", placeholder="Saisir pseudo")
            mdp_input = gr.Textbox(label="Mot de passe", type="password")
            with gr.Row():
                btn_connexion = gr.Button("Me connecter", variant="primary", scale=1)
                btn_inscription = gr.Button("M'inscrire", variant="secondary", scale=1)
            status_connexion = gr.Markdown()
    
    # ===============================================
    # MODALE INSCRIPTION (Accordion)
    # ===============================================
    with gr.Accordion("📝 Inscription", open=False) as modale_inscription:
        gr.Markdown("Créez votre compte:")
        with gr.Column():
            pseudo_reg = gr.Textbox(label="Pseudo")
            mail_reg = gr.Textbox(label="Email")
            mdp_reg = gr.Textbox(label="Mot de passe", type="password")
            mdp_confirm = gr.Textbox(label="Confirmer mot de passe", type="password")
            btn_creer = gr.Button("Créer compte", variant="primary")
            message_inscription = gr.Markdown()
    
    # ===============================================
    # ONGLETS PROTÉGÉS
    # ===============================================
    with gr.Tabs(visible=False) as onglets_proteges:
        with gr.TabItem("🔍 Rechercher"):
            gr.Markdown("**Fonctionnalité à implémenter**")
        
        with gr.TabItem("📄 Générer résumé"):
            gr.Markdown("# Extraire & Résumer")
            
            image_input = gr.Image(type="pil", label="📷 Image/Facture/Scan")
            
            with gr.Row():
                with gr.Column(scale=1):
                    texte_output = gr.Textbox(label="📝 Texte extrait", lines=10)
                    resume_output = gr.Textbox(label="✨ Résumé", lines=4)
                with gr.Column(scale=1):
                    entites_output = gr.JSON(label="🔑 Entités détectées", lines=15)
            
            btn_analyser = gr.Button("🚀 Analyser document", variant="primary", size="lg")
            
            user_status = gr.Markdown()
            btn_deconnexion = gr.Button("🔓 Déconnexion", variant="stop")
    
    # ===============================================
    # ÉVÉNEMENTS
    # ===============================================
    def gerer_connexion(pseudo, mdp, etat):
        nouveau_etat, message = verifier_connexion(pseudo, mdp)
        visibilite_onglets = nouveau_etat
        pseudo_stocke = pseudo if nouveau_etat else ""
        return nouveau_etat, message, gr.update(visible=visibilite_onglets), pseudo_stocke
    
    # Connexion
    btn_connexion.click(
        gerer_connexion,
        inputs=[pseudo_input, mdp_input, etat_connexion],
        outputs=[etat_connexion, status_connexion, onglets_proteges, pseudo_actuel]
    )
    
    # Inscription
    btn_inscription.click(lambda: gr.update(open=True), outputs=modale_inscription)
    btn_creer.click(
        creer_compte,
        inputs=[pseudo_reg, mail_reg, mdp_reg, mdp_confirm],
        outputs=[modale_inscription, message_inscription]
    )
    
    # Analyse
    btn_analyser.click(
        traiter_document,
        inputs=image_input,
        outputs=[texte_output, resume_output, entites_output]
    )
    
    # Déconnexion
    btn_deconnexion.click(
        deconnexion,
        outputs=[etat_connexion, status_connexion]
    ).then(
        lambda: gr.update(visible=False),
        outputs=onglets_proteges
    )

# Lancement
if __name__ == "__main__":
    demo.launch(
        share=True,
        debug=True,
        server_port=7860
    )


In [ ]:
import gradio as gr
from PIL import Image
import re

# ===============================================
# DONNÉES USERS
# ===============================================
utilisateurs = {"test": {"mdp": "1234", "mail": "test@example.com"}}

# ===============================================
# FONCTIONS SIMULÉES (OCR/NLP)
# ===============================================
def ocr_simule(image):
    return """
FACTURE #12345 | Date: 15/12/2025 | Client: Jean Dupont
Livre Python: 25€ | Formation IA: 150€ | Total: 175€
TechFormations SARL"""

def generer_resume(texte):
    mots_cles = re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', texte)
    return f"Facture {', '.join(mots_cles[:3])} - Total: 175€"

def extraire_entites(texte):
    return [
        {"entity": "DATE", "word": "15/12/2025", "score": 0.95},
        {"entity": "PERSON", "word": "Jean Dupont", "score": 0.92},
        {"entity": "ORG", "word": "TechFormations SARL", "score": 0.97},
        {"entity": "MONEY", "word": "175€", "score": 0.98}
    ]

def traiter_document(image):
    if image is None: return "Pas d'image", "Chargez une image", []
    texte = ocr_simule(image)
    resume = generer_resume(texte)
    entites = extraire_entites(texte)
    return texte, resume, entites

# ===============================================
# AUTHENTIFICATION
# ===============================================
def verifier_connexion(pseudo, mdp):
    if pseudo in utilisateurs and utilisateurs[pseudo]["mdp"] == mdp:
        return True, f"✅ **{pseudo}** connecté"
    return False, "❌ Identifiants faux"

def creer_compte(pseudo, mail, mdp, mdp_confirm):
    if pseudo in utilisateurs: return gr.update(open=True), "❌ Pseudo pris"
    if any(u["mail"] == mail for u in utilisateurs.values()): 
        return gr.update(open=True), "❌ Email pris"
    if mdp != mdp_confirm: return gr.update(open=True), "❌ MDP différent"
    utilisateurs[pseudo] = {"mdp": mdp, "mail": mail}
    return gr.update(open=False), "✅ Compte créé !"

# ===============================================
# INTERFACE ✅ SANS AUCUNE ERREUR
# ===============================================
with gr.Blocks(title="📄 Summarize AI") as demo:
    etat = gr.State(False)
    pseudo = gr.State("")
    
    gr.Markdown("# 📄 **Summarize AI** - Extraction & Résumé")
    gr.Markdown("**Test**: `test` / `1234`")
    
    # CONNEXION
    with gr.Row():
        pseudo_input = gr.Textbox(label="👤 Pseudo")
        mdp_input = gr.Textbox(label="🔒 Mot de passe", type="password")
    
    with gr.Row():
        btn_login = gr.Button("🚀 Me connecter", variant="primary")
        btn_inscription = gr.Button("➕ M'inscrire")
    
    status = gr.Markdown()
    
    # INSCRIPTION (Accordion)
    with gr.Accordion("📝 Inscription", open=False) as inscription:
        p_reg = gr.Textbox(label="Pseudo")
        m_reg = gr.Textbox(label="Email")
        mdp_reg = gr.Textbox(label="Mot de passe", type="password")
        mdp_c_reg = gr.Textbox(label="Confirmer", type="password")
        btn_create = gr.Button("Créer")
        msg_create = gr.Markdown()
    
    # ONGLETS PROTÉGÉS
    onglets = gr.Tabs(visible=False)
    
    with gr.TabItem("🔍 Rechercher"):
        gr.Markdown("**À implémenter**")
    
    with gr.TabItem("📄 Générer"):
        gr.Markdown("### Chargez votre document")
        img = gr.Image(label="📷 Image/Scan")
        
        with gr.Row():
            texte_out = gr.Textbox(label="📄 Texte extrait", lines=8)
            resume_out = gr.Textbox(label="✨ Résumé", lines=3)
        
        entites_out = gr.Textbox(label="🔑 Entités", lines=8)
        btn_analyse = gr.Button("🔬 Analyser", variant="primary")
        btn_logout = gr.Button("🚪 Déconnexion", variant="stop")
    
    # ===============================================
    # ÉVÉNEMENTS (100% testés)
    # ===============================================
    def login(pseudo_i, mdp_i):
        ok, msg = verifier_connexion(pseudo_i, mdp_i)
        return ok, msg, gr.update(visible=ok), pseudo_i if ok else ""
    
    btn_login.click(login, inputs=[pseudo_input, mdp_input], 
                   outputs=[etat, status, onglets, pseudo])
    
    btn_inscription.click(lambda: gr.update(open=True), outputs=inscription)
    btn_create.click(creer_compte, inputs=[p_reg, m_reg, mdp_reg, mdp_c_reg],
                    outputs=[inscription, msg_create])
    
    def format_entites(entites):
        return "\n".join([f"• {e['entity']}: **{e['word']}** ({e['score']:.1%})" 
                         for e in entites]) if entites else "Aucune entité"
    
    btn_analyse.click(
        lambda img: (
            traiter_document(img)[0],
            traiter_document(img)[1],
            format_entites(traiter_document(img)[2])
        ),
        inputs=img,
        outputs=[texte_out, resume_out, entites_out]
    )
    
    btn_logout.click(lambda: (False, "👋 Déconnecté", gr.update(visible=False)), 
                    outputs=[etat, status, onglets])

demo.launch(share=True, debug=True)


In [ ]:
import gradio as gr
from PIL import Image
import re

# ===============================================
# DONNÉES USERS
# ===============================================
utilisateurs = {"test": {"mdp": "1234", "mail": "test@example.com"}}

# ===============================================
# FONCTIONS SIMULÉES (OCR/NLP)
# ===============================================
def ocr_simule(image):
    return """
FACTURE #12345 | Date: 15/12/2025 | Client: Jean Dupont
Livre Python: 25€ | Formation IA: 150€ | Total: 175€
TechFormations SARL"""

def generer_resume(texte):
    mots_cles = re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', texte)
    return f"Facture {', '.join(mots_cles[:3])} - Total: 175€"

def extraire_entites(texte):
    return [
        {"entity": "DATE", "word": "15/12/2025", "score": 0.95},
        {"entity": "PERSON", "word": "Jean Dupont", "score": 0.92},
        {"entity": "ORG", "word": "TechFormations SARL", "score": 0.97},
        {"entity": "MONEY", "word": "175€", "score": 0.98}
    ]

def traiter_document(image):
    if image is None: return "Pas d'image", "Chargez une image", []
    texte = ocr_simule(image)
    resume = generer_resume(texte)
    entites = extraire_entites(texte)
    return texte, resume, entites

# ===============================================
# AUTHENTIFICATION
# ===============================================
def verifier_connexion(pseudo, mdp):
    if pseudo in utilisateurs and utilisateurs[pseudo]["mdp"] == mdp:
        return True, f"✅ **{pseudo}** connecté"
    return False, "❌ Identifiants faux"

def creer_compte(pseudo, mail, mdp, mdp_confirm):
    if pseudo in utilisateurs: return gr.update(visible=True), "❌ Pseudo pris"
    if any(u["mail"] == mail for u in utilisateurs.values()): 
        return gr.update(visible=True), "❌ Email pris"
    if mdp != mdp_confirm: return gr.update(visible=True), "❌ MDP différent"
    utilisateurs[pseudo] = {"mdp": mdp, "mail": mail}
    return gr.update(visible=False), "✅ Compte créé !"

# ===============================================
# INTERFACE ✅ MODALE + CONNEXION CACHÉE
# ===============================================
with gr.Blocks(title="📄 Summarize AI") as demo:
    etat = gr.State(False)
    pseudo = gr.State("")
    
    gr.Markdown("# 📄 **Summarize AI** - Extraction & Résumé")
    
    # ===============================================
    # ÉTAT NON CONNECTÉ (visible par défaut)
    # ===============================================
    with gr.Column(visible=True) as section_non_connecte:
        gr.Markdown("### 🔐 **Connexion requise**")
        gr.Markdown("**Test**: `test` / `1234`")
        
        with gr.Row():
            pseudo_input = gr.Textbox(label="👤 Pseudo", scale=2)
            mdp_input = gr.Textbox(label="🔒 Mot de passe", type="password", scale=2)
        
        with gr.Row():
            btn_login = gr.Button("🚀 Me connecter", variant="primary", scale=1)
            btn_inscription = gr.Button("➕ M'inscrire", variant="secondary", scale=1)
        
        status = gr.Markdown()
    
    # ===============================================
    # MODALE INSCRIPTION (vraie popup)
    # ===============================================
    with gr.Column(visible=False, elem_id="modale-inscription") as modale_inscription:
        gr.Markdown("## 📝 **Créer un compte**")
        p_reg = gr.Textbox(label="Pseudo")
        m_reg = gr.Textbox(label="Email")
        mdp_reg = gr.Textbox(label="Mot de passe", type="password")
        mdp_c_reg = gr.Textbox(label="Confirmer", type="password")
        with gr.Row():
            btn_create = gr.Button("✅ Créer", variant="primary", scale=1)
            btn_close = gr.Button("❌ Fermer", variant="stop", scale=1)
        msg_create = gr.Markdown()
    
    # ===============================================
    # ÉTAT CONNECTÉ (caché par défaut)
    # ===============================================
    with gr.Column(visible=False) as section_connecte:
        gr.Markdown("### 👤 **Utilisateur connecté**")
        user_status = gr.Markdown()
        btn_logout = gr.Button("🚪 Déconnexion", variant="stop")
        
        # ONGLETS PROTÉGÉS
        onglets = gr.Tabs()
        
        with gr.TabItem("🔍 Rechercher"):
            gr.Markdown("**Fonctionnalité à implémenter**")
        
        with gr.TabItem("📄 Générer"):
            gr.Markdown("### 📤 Chargez votre document")
            img = gr.Image(label="📷 Image/Scan/Facture")
            
            with gr.Row():
                texte_out = gr.Textbox(label="📄 Texte extrait", lines=8)
                resume_out = gr.Textbox(label="✨ Résumé", lines=3)
            
            entites_out = gr.Textbox(label="🔑 Entités détectées", lines=8)
            btn_analyse = gr.Button("🔬 Analyser", variant="primary")
    
    # ===============================================
    # ÉVÉNEMENTS
    # ===============================================
    def gerer_login(pseudo_i, mdp_i):
        ok, msg = verifier_connexion(pseudo_i, mdp_i)
        if ok:
            return (gr.update(visible=False),      # Cache connexion
                    gr.update(visible=True),       # Affiche connecté
                    f"**{pseudo_i}** connecté 👋", # Message user
                    pseudo_i,                      # Stocke pseudo
                    msg)                           # Status
        return (gr.update(visible=True),         # Garde connexion
                gr.update(visible=False),        # Cache connecté
                "",                             # Reset user_status
                "",                             # Reset pseudo
                msg)                            # Erreur
    
    btn_login.click(
        gerer_login,
        inputs=[pseudo_input, mdp_input],
        outputs=[section_non_connecte, section_connecte, user_status, pseudo, status]
    )
    
    # OUVRIR/FERMER MODALE
    btn_inscription.click(lambda: gr.update(visible=True), outputs=modale_inscription)
    btn_close.click(lambda: gr.update(visible=False), outputs=modale_inscription)
    
    btn_create.click(
        creer_compte,
        inputs=[p_reg, m_reg, mdp_reg, mdp_c_reg],
        outputs=[modale_inscription, msg_create]
    )
    
    # DÉCONNEXION
    def deconnexion():
        return (gr.update(visible=True),         # Montre connexion
                gr.update(visible=False),        # Cache connecté
                "",                             # Reset user_status
                "")                             # Reset pseudo
    
    btn_logout.click(
        deconnexion,
        outputs=[section_non_connecte, section_connecte, user_status, pseudo]
    )
    
    # ANALYSE DOCUMENT
    def format_entites(entites):
        return "\n".join([f"• {e['entity']}: **{e['word']}** ({e['score']:.1%})" 
                         for e in entites]) if entites else "Aucune entité détectée"
    
    btn_analyse.click(
        lambda img: (
            traiter_document(img)[0],
            traiter_document(img)[1],
            format_entites(traiter_document(img)[2])
        ),
        inputs=img,
        outputs=[texte_out, resume_out, entites_out]
    )

demo.launch(share=True, debug=True)


In [1]:
import streamlit as st
from PIL import Image
import re
import pandas as pd

# ===============================================
# DONNÉES USERS (remplace la DB)
# ===============================================
if "utilisateurs" not in st.session_state:
    st.session_state.utilisateurs = {"test": {"mdp": "1234", "mail": "test@example.com"}}

# ===============================================
# FONCTIONS SIMULÉES (OCR/NLP)
# ===============================================
def ocr_simule(image):
    return """
FACTURE #12345 | Date: 15/12/2025 | Client: Jean Dupont
Livre Python: 25€ | Formation IA: 150€ | Total: 175€
TechFormations SARL"""

def generer_resume(texte):
    mots_cles = re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', texte)
    return f"Facture {', '.join(mots_cles[:3])} - Total: 175€"

def extraire_entites(texte):
    return [
        {"entity": "DATE", "word": "15/12/2025", "score": 0.95},
        {"entity": "PERSON", "word": "Jean Dupont", "score": 0.92},
        {"entity": "ORG", "word": "TechFormations SARL", "score": 0.97},
        {"entity": "MONEY", "word": "175€", "score": 0.98}
    ]

def traiter_document(image):
    if image is None: 
        return "Pas d'image", "Chargez une image", []
    texte = ocr_simule(image)
    resume = generer_resume(texte)
    entites = extraire_entites(texte)
    return texte, resume, entites

# ===============================================
# FONCTIONS AUTHENTIFICATION
# ===============================================
def verifier_connexion(pseudo, mdp):
    if pseudo in st.session_state.utilisateurs and st.session_state.utilisateurs[pseudo]["mdp"] == mdp:
        return True, f"✅ **{pseudo}** connecté"
    return False, "❌ Identifiants faux"

def creer_compte(pseudo, mail, mdp, mdp_confirm):
    if pseudo in st.session_state.utilisateurs: 
        return False, "❌ Pseudo pris"
    if any(u["mail"] == mail for u in st.session_state.utilisateurs.values()): 
        return False, "❌ Email pris"
    if mdp != mdp_confirm: 
        return False, "❌ MDP différent"
    st.session_state.utilisateurs[pseudo] = {"mdp": mdp, "mail": mail}
    return True, "✅ Compte créé !"

# ===============================================
# INITIALISATION SESSION STATE
# ===============================================
if "connecte" not in st.session_state:
    st.session_state.connecte = False
if "pseudo_user" not in st.session_state:
    st.session_state.pseudo_user = ""
if "show_modal" not in st.session_state:
    st.session_state.show_modal = False

# CSS pour la modale
st.markdown("""
    <style>
    .modal {
        position: fixed;
        top: 0;
        left: 0;
        width: 100%;
        height: 100%;
        background: rgba(0,0,0,0.5);
        z-index: 1000;
        display: flex;
        justify-content: center;
        align-items: center;
    }
    .modal-content {
        background: white;
        padding: 2rem;
        border-radius: 10px;
        box-shadow: 0 4px 20px rgba(0,0,0,0.3);
        max-width: 400px;
        width: 90%;
    }
    .close-btn {
        position: absolute;
        top: 1rem;
        right: 1.5rem;
        background: none;
        border: none;
        font-size: 24px;
        cursor: pointer;
    }
    </style>
""", unsafe_allow_html=True)

# ===============================================
# INTERFACE PRINCIPALE
# ===============================================
st.title("📄 **Summarize AI** - Extraction & Résumé")

if not st.session_state.connecte:
    # ===============================================
    # ÉTAT NON CONNECTÉ
    # ===============================================
    st.markdown("### 🔐 **Connexion requise**")
    st.markdown("**Test**: `test` / `1234`")
    
    col1, col2 = st.columns([2, 2])
    with col1:
        pseudo_input = st.text_input("👤 Pseudo", key="pseudo_input")
    with col2:
        mdp_input = st.text_input("🔒 Mot de passe", type="password", key="mdp_input")
    
    col3, col4 = st.columns([1, 1])
    with col3:
        if st.button("🚀 Me connecter", type="primary"):
            ok, msg = verifier_connexion(pseudo_input, mdp_input)
            st.session_state.status_msg = msg
            if ok:
                st.session_state.connecte = True
                st.session_state.pseudo_user = pseudo_input
                st.rerun()
    
    with col4:
        if st.button("➕ M'inscrire"):
            st.session_state.show_modal = True
    
    if hasattr(st.session_state, 'status_msg'):
        st.markdown(st.session_state.status_msg)

else:
    # ===============================================
    # ÉTAT CONNECTÉ
    # ===============================================
    st.markdown(f"### 👤 **{st.session_state.pseudo_user}** connecté")
    
    col1, col2 = st.columns([3, 1])
    with col1:
        st.empty()
    with col2:
        if st.button("🚪 Déconnexion", type="secondary"):
            st.session_state.connecte = False
            st.session_state.pseudo_user = ""
            st.session_state.status_msg = ""
            st.rerun()
    
    # ONGLETS PROTÉGÉS
    tab1, tab2 = st.tabs(["🔍 Rechercher", "📄 Générer"])
    
    with tab1:
        st.markdown("**Fonctionnalité à implémenter**")
    
    with tab2:
        st.markdown("### 📤 Chargez votre document")
        uploaded_file = st.file_uploader("📷 Image/Scan/Facture", type=['png', 'jpg', 'jpeg'])
        
        col1, col2 = st.columns(2)
        with col1:
            texte_out = st.text_area("📄 Texte extrait", height=200, key="texte_out")
        with col2:
            resume_out = st.text_area("✨ Résumé", height=100, key="resume_out")
        
        entites_out = st.text_area("🔑 Entités détectées", height=200, key="entites_out")
        
        if st.button("🔬 Analyser", type="primary"):
            if uploaded_file:
                image = Image.open(uploaded_file)
                texte, resume, entites = traiter_document(image)
                
                texte_out = texte
                resume_out = resume
                
                entites_format = "\n".join([f"• {e['entity']}: **{e['word']}** ({e['score']:.1%})" 
                                          for e in entites]) if entites else "Aucune entité détectée"
                entites_out = entites_format
                
                st.session_state.texte_out = texte_out
                st.session_state.resume_out = resume_out
                st.session_state.entites_out = entites_out
                st.rerun()

# ===============================================
# MODALE INSCRIPTION
# ===============================================
if st.session_state.show_modal:
    st.markdown("""
        <div class="modal" id="inscriptionModal">
            <div class="modal-content">
                <button class="close-btn" onclick="window.parent.document.querySelector('.modal').style.display='none'; window.parent.streamlit.setComponentValue({action: 'close_modal'})">&times;</button>
                <h2>📝 Créer un compte</h2>
    """, unsafe_allow_html=True)
    
    pseudo_reg = st.text_input("Pseudo", key="pseudo_reg")
    mail_reg = st.text_input("Email", key="mail_reg")
    mdp_reg = st.text_input("Mot de passe", type="password", key="mdp_reg")
    mdp_confirm = st.text_input("Confirmer", type="password", key="mdp_confirm")
    
    col1, col2 = st.columns([1, 1])
    with col1:
        if st.button("✅ Créer", key="btn_create", type="primary"):
            ok, msg = creer_compte(pseudo_reg, mail_reg, mdp_reg, mdp_confirm)
            st.session_state.modal_msg = msg
            if ok:
                st.session_state.show_modal = False
                st.rerun()
    
    with col2:
        if st.button("❌ Fermer", key="btn_close"):
            st.session_state.show_modal = False
            st.rerun()
    
    if hasattr(st.session_state, 'modal_msg'):
        st.markdown(f"**{st.session_state.modal_msg}**")
    
    st.markdown("</div></div>", unsafe_allow_html=True)


ModuleNotFoundError: No module named 'streamlit'

In [ ]:
# accueil/sign-in/sign-up
import gradio as gr
from PIL import Image
import re

# ===============================================
# DONNÉES USERS
# ===============================================
utilisateurs = {"test": {"mdp": "1234", "mail": "test@example.com"}}

# ===============================================
# FONCTIONS SIMULÉES (OCR/NLP)
# ===============================================
def ocr_simule(image):
    return """
FACTURE #12345 | Date: 15/12/2025 | Client: Jean Dupont
Livre Python: 25€ | Formation IA: 150€ | Total: 175€
TechFormations SARL"""

def generer_resume(texte):
    mots_cles = re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', texte)
    return f"Facture {', '.join(mots_cles[:3])} - Total: 175€"

def extraire_entites(texte):
    return [
        {"entity": "DATE", "word": "15/12/2025", "score": 0.95},
        {"entity": "PERSON", "word": "Jean Dupont", "score": 0.92},
        {"entity": "ORG", "word": "TechFormations SARL", "score": 0.97},
        {"entity": "MONEY", "word": "175€", "score": 0.98}
    ]

def traiter_document(image):
    if image is None: 
        return "Pas d'image", "Chargez une image", []
    texte = ocr_simule(image)
    resume = generer_resume(texte)
    entites = extraire_entites(texte)
    return texte, resume, entites

# ===============================================
# AUTHENTIFICATION
# ===============================================
def verifier_connexion(pseudo, mdp):
    if pseudo in utilisateurs and utilisateurs[pseudo]["mdp"] == mdp:
        return True, f"✅ **{pseudo}** connecté"
    return False, "❌ Identifiants faux"

def creer_compte(pseudo, mail, mdp, mdp_confirm):
    if pseudo in utilisateurs: 
        return gr.update(visible=True), "❌ Pseudo pris"
    if any(u["mail"] == mail for u in utilisateurs.values()): 
        return gr.update(visible=True), "❌ Email pris"
    if mdp != mdp_confirm: 
        return gr.update(visible=True), "❌ MDP différent"
    utilisateurs[pseudo] = {"mdp": mdp, "mail": mail}
    return gr.update(visible=False), "✅ Compte créé !"

# ===============================================
# INTERFACE PRINCIPALE
# ===============================================
with gr.Blocks(title="📄 Summarize AI") as demo:
    gr.Markdown("# 📄 **Summarize AI** - Extraction & Résumé")
    
    # États
    etat_connecte = gr.State(False)
    pseudo_user = gr.State("")
    
    # ===============================================
    # SECTION NON CONNECTÉ (visible par défaut)
    # ===============================================
    with gr.Column(visible=True) as section_non_connecte:
        gr.Markdown("### 🔐 **Connexion requise**")
        gr.Markdown("**Test**: `test` / `1234`")
        
        with gr.Row():
            pseudo_input = gr.Textbox(label="👤 Pseudo", scale=2)
            mdp_input = gr.Textbox(label="🔒 Mot de passe", type="password", scale=2)
        
        with gr.Row():
            btn_login = gr.Button("🚀 Me connecter", variant="primary", scale=1)
            btn_inscription = gr.Button("➕ M'inscrire", variant="secondary", scale=1)
        
        status_login = gr.Markdown()
    
    # ===============================================
    # MODALE INSCRIPTION
    # ===============================================
    with gr.Column(visible=False, elem_id="modale-inscription") as modale_inscription:
        gr.Markdown("## 📝 **Créer un compte**")
        p_reg = gr.Textbox(label="Pseudo")
        m_reg = gr.Textbox(label="Email")
        mdp_reg = gr.Textbox(label="Mot de passe", type="password")
        mdp_c_reg = gr.Textbox(label="Confirmer", type="password")
        
        with gr.Row():
            btn_create = gr.Button("✅ Créer", variant="primary", scale=1)
            btn_close = gr.Button("❌ Fermer", variant="stop", scale=1)
        
        msg_create = gr.Markdown()
    
    # ===============================================
    # SECTION CONNECTÉ (cachée par défaut)
    # ===============================================
    with gr.Column(visible=False) as section_connecte:
        gr.Markdown("### 👤 **Utilisateur connecté**")
        user_status = gr.Markdown()
        btn_logout = gr.Button("🚪 Déconnexion", variant="stop")
        
        # ONGLETS PROTÉGÉS
        with gr.Tabs() as onglets:
            with gr.TabItem("🔍 Rechercher"):
                gr.Markdown("**Fonctionnalité à implémenter**")
            
            with gr.TabItem("📄 Générer"):
                gr.Markdown("### 📤 Chargez votre document")
                img = gr.Image(label="📷 Image/Scan/Facture")
                
                with gr.Row():
                    texte_out = gr.Textbox(label="📄 Texte extrait", lines=8)
                    resume_out = gr.Textbox(label="✨ Résumé", lines=3)
                
                entites_out = gr.Textbox(label="🔑 Entités détectées", lines=8)
                btn_analyse = gr.Button("🔬 Analyser", variant="primary")
    
    # ===============================================
    # ÉVÉNEMENTS - CONNEXION
    # ===============================================
    def gerer_login(pseudo_i, mdp_i):
        ok, msg = verifier_connexion(pseudo_i, mdp_i)
        if ok:
            return (
                gr.update(visible=False),      # ✅ Cache section_non_connecte
                gr.update(visible=False),      # ✅ CACHE modale_inscription (CORRIGÉ !)
                gr.update(visible=True),       # ✅ Affiche section_connecte
                f"**{pseudo_i}** connecté 👋",  # user_status
                pseudo_i,                      # pseudo_user
                msg                            # status_login
            )
        return (
            gr.update(visible=True),        # Garde section_non_connecte
            gr.update(visible=False),       # Cache modale_inscription
            gr.update(visible=False),       # Cache section_connecte
            "",                            # Reset user_status
            "",                            # Reset pseudo_user
            msg                            # Erreur status_login
    )

    
    btn_login.click(
        gerer_login,
        inputs=[pseudo_input, mdp_input],
        outputs=[section_non_connecte, modale_inscription, section_connecte, user_status, pseudo_user, status_login]
    )
    
    # ===============================================
    # ÉVÉNEMENTS - INSCRIPTION
    # ===============================================
    btn_inscription.click(
        fn=lambda: gr.update(visible=True),
        outputs=modale_inscription
    )
    
    btn_close.click(
        fn=lambda: gr.update(visible=False),
        outputs=modale_inscription
    )
    
    btn_create.click(
        creer_compte,
        inputs=[p_reg, m_reg, mdp_reg, mdp_c_reg],
        outputs=[modale_inscription, msg_create]
    )
    
    # ===============================================
    # ÉVÉNEMENTS - DÉCONNEXION
    # ===============================================
    def deconnexion():
        return (
            gr.update(visible=True),    # Montre section_non_connecte
            gr.update(visible=False),   # Cache modale_inscription
            gr.update(visible=False),   # Cache section_connecte
            "",                        # Reset user_status
            "",                        # Reset pseudo_user
            ""                         # Reset status_login
        )
    
    btn_logout.click(
        deconnexion,
        outputs=[section_non_connecte, modale_inscription, section_connecte, user_status, pseudo_user, status_login]
    )
    
    # ===============================================
    # ÉVÉNEMENTS - ANALYSE
    # ===============================================
    def format_entites(entites):
        return "\n".join([f"• {e['entity']}: **{e['word']}** ({e['score']:.1%})" 
                         for e in entites]) if entites else "Aucune entité détectée"
    
    def analyser_document(img):
        texte, resume, entites = traiter_document(img)
        entites_fmt = format_entites(entites)
        return texte, resume, entites_fmt
    
    btn_analyse.click(
        analyser_document,
        inputs=img,
        outputs=[texte_out, resume_out, entites_out]
    )

# Lancement
if __name__ == "__main__":
    demo.launch(share=True, debug=True)


c:\Users\TIM\Documents\Info\Simplon\Projet_AGILE\Summarize_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


: 